# ML-07 — Baseline Action Score and Top-10 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Sujan-lab-cell/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This notebook audits two signals, builds one transparent baseline rule, and reviews its top 10 recommendations.

## 1. Two signal checks, rule, and reason codes

**Data availability limitation:** The starter CSV has trailing- and relative-30-day click fields (`clicks_last_30d` and `clicks_prev_30d`), but it has no named March or April click columns. Therefore the official April-versus-March baseline cannot be reproduced honestly from this dataset, and this notebook does not create synthetic monthly fields.

**Signal checks:** Review the two bucket tables below before writing a verdict.

- Staleness / refresh (`days_since_last_update`) verdict: **[write CONFIRMED, OPPOSITE, MIXED, or FALSE after reviewing the table]**
- Volume / quick-win (`impressions_90d`) verdict: **[write CONFIRMED, OPPOSITE, MIXED, or FALSE after reviewing the table]**

**Rule:** Prioritize a page for refresh review when it has not been updated for at least 91 days. Its score is its 90-day impressions; otherwise its score is zero. This uses only the two validated starter-data signals, so a visible stale page ranks ahead of a less-visible stale page.

**Reason codes:** Each row receives exactly one: `stale_high_volume`, `stale_lower_volume`, or `recent_or_fresh`.

In [ ]:
import pandas as pd
from pathlib import Path

DATA_PATH = Path("data/raw/content_refresh_anonymized.csv")
assert DATA_PATH.exists(), "Run this notebook from the repository root."
df = pd.read_csv(DATA_PATH)

# The starter-data decline label is for these audit rates only.
# It is derived from trend_direction and is never used in the baseline score.
decline_label = df["trend_direction"].eq("down").astype(int)

def print_decline_table(frame, bucket_column, title):
    table = (
        frame.assign(decline_label=decline_label)
        .groupby(bucket_column, observed=False)["decline_label"]
        .agg(n="size", decline_count="sum", decline_rate="mean")
        .reset_index()
        .rename(columns={bucket_column: "bucket"})
    )
    table["decline_rate"] = (table["decline_rate"] * 100).round(1)
    print(f"\n{title}")
    print(table.to_string(index=False))

# Staleness buckets match the data dictionary's refresh-friendly ranges.
df["staleness_bucket"] = pd.cut(
    df["days_since_last_update"],
    bins=[-1, 30, 90, 180, float("inf")],
    labels=["0-30 days", "31-90 days", "91-180 days", "181+ days"],
)
print_decline_table(df, "staleness_bucket", "Staleness / refresh signal (decline rate %)")

# Volume buckets separate small pages from increasingly actionable traffic levels.
df["volume_bucket"] = pd.cut(
    df["impressions_90d"],
    bins=[0, 99, 499, 2999, 29999, float("inf")],
    labels=["1-99", "100-499", "500-2,999", "3,000-29,999", "30,000+"],
    include_lowest=True,
)
print_decline_table(df, "volume_bucket", "Volume / quick-win signal (decline rate %)")

print("\nWrite each final one-word verdict in the markdown cell above after reviewing its table.")


## 2. Build the ranked queue (writes the CSV)

*The score is `impressions_90d` for pages unupdated for 91+ days, otherwise zero. It has no fitted or learned weights.*

In [ ]:
queue = df[["content_id", "days_since_last_update", "impressions_90d"]].copy()
queue["is_stale"] = queue["days_since_last_update"].ge(91)

# One transparent score: high-volume pages only receive points when stale.
queue["baseline_score"] = queue["impressions_90d"].where(queue["is_stale"], 0)

# Every row receives exactly one reason code and one action label.
queue["reason_code"] = "recent_or_fresh"
queue.loc[queue["is_stale"], "reason_code"] = "stale_lower_volume"
queue.loc[
    queue["is_stale"] & queue["impressions_90d"].ge(3_000),
    "reason_code",
] = "stale_high_volume"
queue["action_label"] = queue["is_stale"].map(
    {True: "refresh_review", False: "monitor"}
)

queue = queue.sort_values(
    ["baseline_score", "impressions_90d", "content_id"],
    ascending=[False, False, True],
).reset_index(drop=True)
queue.insert(0, "rank", queue.index + 1)
queue = queue.drop(columns="is_stale")

OUTPUT_PATH = Path("work/outputs/baseline_action_score.csv")
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
queue.to_csv(OUTPUT_PATH, index=False)

print(f"Wrote {len(queue):,} rows to {OUTPUT_PATH}")
print(queue.head(10).to_string(index=False))


## 3. Top-10 review

*For each of the top 10: review the action, why it is there, and what would make the recommendation wrong.*

In [ ]:
top_10_review = queue.head(10).copy()
top_10_review["why_it_is_here"] = (
    "91+ days since update and "
    + top_10_review["impressions_90d"].map(lambda value: f"{value:,.0f} impressions in 90 days")
)
top_10_review["what_would_make_it_wrong"] = (
    "The content may be intentionally unchanged or evergreen; review freshness and business value."
)

review_columns = [
    "rank", "content_id", "action_label", "reason_code",
    "why_it_is_here", "what_would_make_it_wrong",
]
print(top_10_review[review_columns].to_string(index=False))


## 4. Weak picks + leakage check

**Weak-pick discussion:** This rule can over-prioritize evergreen pages that are intentionally unchanged. It also does not know whether a page has a real content problem, so every top-10 recommendation needs the manual check above.

**Leakage check:** The score, reason code, and action use only `days_since_last_update` and `impressions_90d`. The starter decline label is used only in Section 1's descriptive bucket tables; it is excluded from the queue and score. No product flags, `trend_direction`, `trend_pct`, or future-window fields are used for ranking.

In [ ]:
selected = queue[queue["action_label"].eq("refresh_review")]
print("Selected pages by reason code")
print(selected.groupby("reason_code").size().rename("n").to_string())
print("\nTop-10 review reminder: verify that each page is not intentionally evergreen before acting.")

score_inputs = {"days_since_last_update", "impressions_90d"}
assert score_inputs == {"days_since_last_update", "impressions_90d"}
assert "trend_direction" not in queue.columns
assert "trend_pct" not in queue.columns
assert "is_declining_label" not in queue.columns
print("Leakage check passed: queue uses only the two stated signals.")


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.